# DEH 30-Day PySpark Challenge
## Day 28 — Practice Set 7: Hard Multi-Step Pipelines

**Instructions:**
1. `File → Save a copy in Drive` first
2. Break each problem into numbered steps BEFORE coding
3. Reference solutions follow each problem

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder.appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id", StringType(), True), StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True), StructField("order_date", DateType(), True),
    StructField("quantity", IntegerType(), True), StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", IntegerType(), True), StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True), StructField("region", StringType(), True)
])
customers_schema = StructType([
    StructField("customer_id", StringType(), True), StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True), StructField("email", StringType(), True),
    StructField("city", StringType(), True), StructField("state", StringType(), True),
    StructField("country", StringType(), True), StructField("signup_date", DateType(), True),
    StructField("segment", StringType(), True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")

print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()}")

---
## Problem 1 (Hard) — Customer retention by segment

Retained = ordered in month AND previous month. Retention rate per segment per month transition.

In [ ]:
# Write your numbered plan as comments, then attempt


### Reference Solution — Problem 1

**Plan:**
1. Get distinct (customer_id, year_month) pairs from orders
2. Self-join each row to the same customer in the NEXT month using `lead()` style logic via a generated next-month column
3. A customer is retained in month M if they also ordered in month M-1
4. Join with customers to get segment
5. Group by segment + month, calculate retention %

In [ ]:
# Step 1 — distinct customer-month activity
activity = orders_df.withColumn("year_month", F.date_format("order_date", "yyyy-MM")) \
    .select("customer_id", "year_month").distinct()

# Step 2 — generate the "previous month" for each activity row
activity_prev = activity.withColumn(
    "prev_month",
    F.date_format(F.add_months(F.to_date(F.concat(F.col("year_month"), F.lit("-01"))), -1), "yyyy-MM")
)

# Step 3 — self join: was this customer active in prev_month too?
a = activity.alias("a")
b = activity_prev.alias("b")

retained = b.join(
    a,
    (F.col("a.customer_id") == F.col("b.customer_id")) & (F.col("a.year_month") == F.col("b.prev_month")),
    how="left"
).select(
    F.col("b.customer_id").alias("customer_id"),
    F.col("b.year_month").alias("year_month"),
    F.col("a.customer_id").isNotNull().alias("is_retained")
)

# Step 4 — join with customers for segment
retained_with_segment = retained.join(
    customers_df.select("customer_id", "segment"), on="customer_id", how="inner"
)

# Step 5 — retention rate per segment + month
retained_with_segment.groupBy("segment", "year_month").agg(
    F.count("customer_id").alias("active_customers"),
    F.round(100.0 * F.sum(F.col("is_retained").cast("int")) / F.count("customer_id"), 1).alias("retention_pct")
).orderBy("segment", "year_month").show(30)

---
## Problem 2 (Hard) — Product affinity pair list

For customers with 2+ distinct products, find product pairs they bought. Count of customers per pair.

In [ ]:
# Write your numbered plan as comments, then attempt


### Reference Solution — Problem 2

**Plan:**
1. Get distinct (customer_id, product_id) pairs
2. Self-join on customer_id where product_id_a < product_id_b (alphabetical, avoids duplicates and self-pairs)
3. Group by the product pair, count distinct customers

In [ ]:
customer_products = orders_df.select("customer_id", "product_id").distinct()

a = customer_products.alias("a")
b = customer_products.alias("b")

pairs = a.join(
    b,
    (F.col("a.customer_id") == F.col("b.customer_id")) & (F.col("a.product_id") < F.col("b.product_id")),
    how="inner"
).select(
    F.col("a.customer_id").alias("customer_id"),
    F.col("a.product_id").alias("product_a"),
    F.col("b.product_id").alias("product_b")
)

pairs.groupBy("product_a", "product_b").agg(
    F.countDistinct("customer_id").alias("customer_count")
).orderBy(F.col("customer_count").desc()).show(15)

---
## Problem 3 (Hard) — Rolling 3-order average price per customer

Current + 2 preceding orders averaged. Fewer than 3 → average what exists.

In [ ]:
# Write your numbered plan as comments, then attempt


### Reference Solution — Problem 3

`rowsBetween(-2, 0)` means "2 rows before to current row" — exactly a 3-row rolling window. Spark automatically uses fewer rows at the start of each partition.

In [ ]:
rolling_window = Window.partitionBy("customer_id").orderBy("order_date").rowsBetween(-2, 0)

orders_df.withColumn(
    "rolling_avg_3", F.round(F.avg("unit_price").over(rolling_window), 2)
).select("customer_id", "order_date", "unit_price", "rolling_avg_3") \
 .orderBy("customer_id", "order_date") \
 .show(15)

---
## Problem 4 (Hard) — Identify at-risk customers

Last-3-orders avg < First-3-orders avg, AND total orders >= 5.

In [ ]:
# Write your numbered plan as comments, then attempt


### Reference Solution — Problem 4

**Plan:**
1. Rank each customer's orders both ascending and descending by date
2. Filter for rank <= 3 in each direction to get "first 3" and "last 3" order sets
3. Average each set separately
4. Join the two averages together, filter where last < first AND total orders >= 5

In [ ]:
w_asc  = Window.partitionBy("customer_id").orderBy("order_date")
w_desc = Window.partitionBy("customer_id").orderBy(F.col("order_date").desc())

ranked = orders_df.withColumn("rn_asc", F.row_number().over(w_asc)) \
                   .withColumn("rn_desc", F.row_number().over(w_desc))

# Total orders per customer
order_counts = orders_df.groupBy("customer_id").agg(F.count("order_id").alias("total_orders"))

first_3_avg = ranked.filter(F.col("rn_asc") <= 3) \
    .groupBy("customer_id").agg(F.round(F.avg("unit_price"), 2).alias("first_3_avg"))

last_3_avg = ranked.filter(F.col("rn_desc") <= 3) \
    .groupBy("customer_id").agg(F.round(F.avg("unit_price"), 2).alias("last_3_avg"))

at_risk = first_3_avg.join(last_3_avg, on="customer_id", how="inner") \
    .join(order_counts, on="customer_id", how="inner") \
    .filter((F.col("last_3_avg") < F.col("first_3_avg")) & (F.col("total_orders") >= 5))

at_risk.orderBy(F.col("total_orders").desc()).show()

---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*